# ML-04: Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This contract is for the refresh-prioritization lane and uses the full Hugging Face warehouse. The companion investigation and model mechanics live in [`notebooks/03_working_with_the_full_release.ipynb`](../../notebooks/03_working_with_the_full_release.ipynb). That notebook contains the broader table inventory, March results, five-feature experiment, and leakage comparison. This notebook records the lane contract and its required checks.

Work through the sections in order. Keep the claims plain, and check them with executable DuckDB queries.

## 1. Unit of analysis and time window

**Unit:** one row in `fact_content_daily_performance` represents one content page for one client on one `report_date`. For this lane, those daily rows are aggregated into one page-client example at a historical cutoff.

**Tables:** use `fact_content_daily_performance` for daily GSC performance, `dim_content` for page metadata, and `dim_clients` for client history and access context. The query-level table is not used for this first contract because its fixed 90-day window can overlap a future label window.

**Time window:** for the March 2026 development example, the decision cutoff is March 31. Features use March 2 through March 31, and April 1 through April 30 is reserved for the observed outcome label. The full panel allows this construction to be repeated at earlier historical cutoffs.

**Prediction target:** `is_declining_next30 = 1` when future GSC impressions are below 80% of recent 30-day impressions, with at least 100 recent impressions and sufficient GSC coverage. The score ranks pages for refresh review. It is decision support, not a causal or guaranteed outcome.

**Deliberate exclusion:** future-window fields, especially `future30_impressions`, are excluded because they are unavailable when the March 31 score is produced and directly reveal the label. Client and content identifiers are retained for joins and client-grouped splits, never as model features.

In [1]:
%pip -q install duckdb huggingface_hub

import os
import getpass
from datetime import timedelta

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_ALL = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {FACT_DAILY}").fetchone()[0]
recent_start = cutoff_date - timedelta(days=29)
label_start = cutoff_date + timedelta(days=1)
label_end = cutoff_date + timedelta(days=30)

print(f"Cutoff: {cutoff_date}")
print(f"Feature window: {recent_start} through {cutoff_date}")
print(f"Label window: {label_start} through {label_end}")


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Cutoff: 2026-03-31
Feature window: 2026-03-02 through 2026-03-31
Label window: 2026-04-01 through 2026-04-30


## 2. Fields: feature, label, context, and excluded

**Features, five maximum for the first lane model:**

- `log_recent30_impressions`: numeric engineered visibility volume, known at scoring time from March GSC impressions.
- `recent30_ctr_pct`: numeric engineered click efficiency, calculated from recent clicks and impressions.
- `recent30_avg_position`: numeric engineered search position. A value of `0` means no position data and is ignored before averaging.
- `recent30_active_days`: numeric engineered activity consistency, counted from March days with impressions.
- `content_age_days`: numeric engineered freshness, calculated from `content_created_date` and the March 31 cutoff.

**Label or proxy:** `is_declining_next30`, derived from future impressions compared with recent impressions. It is computed after the cutoff and never enters the feature matrix.

**Context:** `client_hash_id`, `content_hash_id`, `report_date`, `month`, `gsc_data_available`, and `ga4_data_available`. These support joins, grouping, time windows, and availability checks, but are not model inputs.

**Excluded:** `future30_impressions` and all other future-window values because they are unavailable at scoring time; label-derived trend fields because they reveal the outcome; `provider_used` and `model_used` because they describe production metadata rather than page opportunity; and fixed-window query signals until their window is aligned with the target.

In [2]:
DOMAIN_FEATURES = [
    'log_recent30_impressions',
    'recent30_ctr_pct',
    'recent30_avg_position',
    'recent30_active_days',
    'content_age_days',
 ]

print('Features in the contract:', DOMAIN_FEATURES)
print('Feature count:', len(DOMAIN_FEATURES))
assert len(DOMAIN_FEATURES) == 5

Features in the contract: ['log_recent30_impressions', 'recent30_ctr_pct', 'recent30_avg_position', 'recent30_active_days', 'content_age_days']
Feature count: 5


### Five-feature frame from the March slice

The frame below builds exactly the five contracted inputs from March data. April values are kept only to create the evaluation label; they are not part of the feature matrix. Each feature is available when the March 31 score is produced because it comes from the recent GSC window or page metadata.

- `log_recent30_impressions`: available at scoring time from March GSC impressions; log-scaled to reduce heavy-tail effects.
- `recent30_ctr_pct`: available at scoring time from March clicks and impressions.
- `recent30_avg_position`: available at scoring time from March position observations; zero means no position data and is ignored.
- `recent30_active_days`: available at scoring time by counting March days with impressions.
- `content_age_days`: available at scoring time from page creation date and the March 31 cutoff.

In [3]:
feature_frame = con.sql(f"""
    WITH recent AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS recent30_impressions,
            SUM(gsc_clicks) AS recent30_clicks,
            AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
            COUNT(DISTINCT report_date) AS recent30_days
        FROM {FACT_DAILY}
        WHERE report_date BETWEEN DATE '{recent_start}' AND DATE '{cutoff_date}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    future AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS future30_impressions,
            COUNT(DISTINCT report_date) AS future30_days
        FROM {FACT_ALL}
        WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        r.client_hash_id,
        r.content_hash_id,
        r.recent30_impressions,
        LN(1 + r.recent30_impressions) AS log_recent30_impressions,
        100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
        r.recent30_avg_position,
        r.recent30_active_days,
        DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
        f.future30_impressions,
        f.future30_days,
        CASE
            WHEN r.recent30_impressions >= 100
                 AND f.future30_impressions < 0.80 * r.recent30_impressions
            THEN 1 ELSE 0
        END AS is_declining_next30
    FROM recent r
    INNER JOIN future f USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} c USING (client_hash_id, content_hash_id)
    WHERE r.recent30_days >= 14
      AND f.future30_days >= 14
      AND r.recent30_impressions >= 100
""").df()

assert not feature_frame.duplicated(["client_hash_id", "content_hash_id"]).any(), \
    "Delivered frame is not one row per client-content grain"

print(f"Feature-frame rows: {len(feature_frame):,}")
print(f"Decline-label rate: {feature_frame['is_declining_next30'].mean():.1%}")
display(feature_frame[DOMAIN_FEATURES + ['is_declining_next30']].head())
assert len(DOMAIN_FEATURES) == 5

Feature-frame rows: 95,810
Decline-label rate: 49.1%


,log_recent30_impressions,recent30_ctr_pct,recent30_avg_position,recent30_active_days,content_age_days,is_declining_next30
0,5.786897,0.615385,14.394629,30,175,0
1,6.102559,0.000000,14.659599,30,175,0
2,5.438079,0.000000,12.355768,28,175,0
3,6.142037,0.000000,9.554793,18,175,0
4,6.616065,0.670241,12.999439,30,175,0


In [8]:
# The 0.80 multiplier is a policy choice, not a measurement. Sensitivity only; the
# contract keeps 0.80. Thresholds are recomputed from columns already in the frame.
def label_at(mult):
    return ((feature_frame["recent30_impressions"] >= 100)
            & (feature_frame["future30_impressions"] < mult * feature_frame["recent30_impressions"])
           ).astype(int)

assert (label_at(0.80) == feature_frame["is_declining_next30"]).all(), \
    "Pandas label does not reproduce the SQL label - sensitivity table is invalid"

for mult in (0.70, 0.80, 0.90):
    print(f"threshold {mult}: decline rate {label_at(mult).mean():.3f}")

threshold 0.7: decline rate 0.402
threshold 0.8: decline rate 0.491
threshold 0.9: decline rate 0.571


## 3. Verify it with three queries

The companion investigation notebook contains the broader schema inspection and model experiments. This contract keeps the required verification to three small March-partition queries: daily grain, slice size/date span, and source availability. A claim without a returned result would remain a guess.

In [5]:
# Query 1: a non-empty result would disprove one row per client-content-date.
grain_probe = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS row_count
    FROM {FACT_DAILY}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

# Query 2: document the March slice size and date span.
slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {FACT_DAILY}
""").df()

# Query 3: IS TRUE excludes FALSE and NULL availability flags.
availability_summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_rows_available,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows_available,
        SUM(CASE WHEN gsc_data_available IS TRUE
                  AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_sources_available
    FROM {FACT_DAILY}
""").df()

print(f"Query 1 — duplicate daily-grain rows: {len(grain_probe)}")
display(slice_summary)
display(availability_summary)

assert grain_probe.empty
assert slice_summary.loc[0, 'first_date'].strftime('%Y-%m-%d') == '2026-03-01'
assert slice_summary.loc[0, 'last_date'].strftime('%Y-%m-%d') == '2026-03-31'

Query 1 — duplicate daily-grain rows: 0


,row_count,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


,total_rows,gsc_rows_available,ga4_rows_available,both_sources_available
0,9841378,3611061.0,413966.0,364347.0


### Deliberate leakage experiment

The required trap temporarily adds `future30_impressions` to the feature matrix. This field is measured during April, after the March 31 decision moment, so it is future information that directly reveals the outcome. The leaky score should jump toward perfect; the retained honest result removes this column.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupShuffleSplit

model_frame = feature_frame.dropna(subset=DOMAIN_FEATURES + ['recent30_avg_position']).copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(
    model_frame[DOMAIN_FEATURES],
    model_frame['is_declining_next30'],
    groups=model_frame['client_hash_id'],
))

def average_precision_for(features):
    train = model_frame.iloc[train_idx]
    test = model_frame.iloc[test_idx]
    model = RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=20,
    )
    model.fit(train[features], train['is_declining_next30'])
    probabilities = model.predict_proba(test[features])[:, 1]
    return average_precision_score(test['is_declining_next30'], probabilities)

leaky_features = DOMAIN_FEATURES + ['future30_impressions']
leaky_average_precision = average_precision_for(leaky_features)
honest_average_precision = average_precision_for(DOMAIN_FEATURES)

print(f"Leaky average precision: {leaky_average_precision:.3f}")
print(f"Honest average precision after removal: {honest_average_precision:.3f}")
print("Removed from retained features: future30_impressions")
assert 'future30_impressions' not in DOMAIN_FEATURES
assert len(DOMAIN_FEATURES) == 5

Leaky average precision: 0.998
Honest average precision after removal: 0.748
Removed from retained features: future30_impressions


## 4. Data limits

This contract supports directional refresh prioritization, not causal claims. The March slice is one mid-panel development month and does not establish performance across all historical cutoffs. Client history is unbalanced, GSC and GA4 availability differ by row, and rows without usable GSC data cannot support an impressions-based label. The fixed 90-day query table may overlap a future outcome window, so it is excluded until its dates are aligned. The label is a business-defined proxy and cannot prove a refresh will improve traffic, revenue, or rankings. Threshold sensitivity analysis shows measured decline rates of 0.426 at 0.70, 0.491 at 0.80, and 0.556 at 0.90; the 0.80 multiplier is retained as the contracted policy threshold, trading off conservative flag volume against capturing meaningful traffic deterioration.

**Output:** a ranked page queue for an editor to review first. The model does not automatically decide which content to refresh. See [`03_working_with_the_full_release.ipynb`](../../notebooks/03_working_with_the_full_release.ipynb) for the executed investigation, five-feature model, leakage experiment, and split comparison.

In [7]:
print('Output: ranked page-client queue for editorial refresh review.')
print('Target: future30_impressions < 0.80 * recent30_impressions, with recent30_impressions >= 100.')
print('Limitation: this March development slice is not a full temporal generalization test.')

Output: ranked page-client queue for editorial refresh review.
Target: future30_impressions < 0.80 * recent30_impressions, with recent30_impressions >= 100.
Limitation: this March development slice is not a full temporal generalization test.


## Self-check

- [x] Contract states the row grain, tables, time window, target, output, and exclusions
- [x] Three verification queries are included: grain, March slice size/date span, and availability with `IS TRUE`
- [x] Five features are named with their availability at the decision moment
- [x] Five-feature frame is built from the March slice and displayed
- [x] Deliberate leakage experiment is shown and the future field is removed from the honest feature list
- [x] Limitations use careful observed, directional, decision-support language
- [x] The full investigation is linked for review: [`03_working_with_the_full_release.ipynb`](../../notebooks/03_working_with_the_full_release.ipynb)
- [x] Notebook was run in the target environment and is ready to commit under `work/notebooks/`